# Phân tích Big Data cho danh mục GrabMart
**Tác giả:** Phan Thị Hà  
**Nguồn:** `plasma-renderer-507213-p8.grabmart_business`

Notebook tái thực hiện các bảng, kiểm định và biểu đồ trong báo cáo từ 8 bảng dữ liệu; không gán nhãn thủ công.

In [ ]:
# Bước 1 - Chuẩn bị Colab và tải mã nguồn
import os, sys, subprocess
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    subprocess.run(['git','clone','-q','https://github.com/phanthiha/grabmart-business-copilot.git'], check=False)
    os.chdir('/content/grabmart-business-copilot')
    # Không cài lại NumPy/SciPy đang được Colab nạp để tránh xung đột nhị phân.
    subprocess.run([sys.executable,'-m','pip','install','-q','google-cloud-bigquery','db-dtypes'], check=True)
print('Thư mục chạy:', os.getcwd())

In [ ]:
# Bước 2 - Xác thực Google và đọc 8 bảng BigQuery
from pathlib import Path
import pandas as pd, numpy as np
from IPython.display import display
PROJECT_ID='plasma-renderer-507213-p8'; DATASET_ID='grabmart_business'
TABLE_NAMES=['sales_daily','menu_sales','offers','peak_hours','product_scoring','miwi_item_breakdown','miwi_heatmap','customer_reviews']
LOCAL=Path(r'C:/paper/data_b/dulieu12-08/bigquery_ready_complete')
if LOCAL.exists():
    tables={n:pd.read_csv(LOCAL/f'{n}.csv') for n in TABLE_NAMES}; print('Nguồn: CSV kiểm thử')
else:
    from google.colab import auth; auth.authenticate_user()
    from google.cloud import bigquery
    client=bigquery.Client(project=PROJECT_ID)
    tables={n:client.query(f'SELECT * FROM `{PROJECT_ID}.{DATASET_ID}.{n}`').to_dataframe() for n in TABLE_NAMES}
    print('Nguồn: BigQuery')
sales=tables['sales_daily']; offers=tables['offers']; score=tables['product_scoring']; miwi=tables['miwi_item_breakdown']; reviews=tables['customer_reviews']
sales['date']=pd.to_datetime(sales.date); offers['date']=pd.to_datetime(offers.date)
overview=pd.DataFrame([[n,len(d),len(d.columns),int(d.isna().sum().sum())] for n,d in tables.items()],columns=['Bảng','Dòng','Cột','Ô thiếu'])
display(overview)

In [ ]:
# Bước 3 - Tính các bảng kết quả bằng mã nguồn công khai
from src.paper_experiments import concentration, frequency_bands, assortment_performance, quality_summary, promotion_summary, cramer_test, duplicate_test, logistic_cross_validation, sensitivity, operational_extension
business=pd.DataFrame([['Ngày quan sát',sales.date.nunique()],['Doanh thu gộp',sales.gross_sales_vnd.sum()],['Doanh thu thuần',sales.net_sales_vnd.sum()],['Giao dịch',sales.transaction_count.sum()],['AOV',sales.gross_sales_vnd.sum()/sales.transaction_count.sum()]],columns=['Chỉ tiêu','Giá trị'])
conc=concentration(score); bands=frequency_bands(score)
category=assortment_performance(score,'product_group'); price=assortment_performance(score,'price_segment')
quality=quality_summary(score); promo,promo_by_offer=promotion_summary(offers,sales)
tests=pd.DataFrame([cramer_test(score,'product_group'),cramer_test(score,'price_segment'),duplicate_test(score)])
cv=logistic_cross_validation(score); sens=sensitivity(score); ops=operational_extension(miwi,reviews)
display(business); display(conc); display(bands); display(category); display(price); display(quality); display(promo); display(tests); display(cv); display(sens); display(ops)

In [ ]:
# Bước 4 - Sinh các biểu đồ sử dụng trong báo cáo
import plotly.express as px, plotly.graph_objects as go
daily=sales.groupby('date',as_index=False).gross_sales_vnd.sum()
px.line(daily,x='date',y='gross_sales_vnd',markers=True,title='Doanh thu gộp theo ngày').show()
sold=score.loc[score.recorded_sales.astype(bool)].sort_values('gross_revenue',ascending=False).copy(); sold['Tích lũy']=sold.gross_revenue.cumsum()/sold.gross_revenue.sum()
px.line(sold.reset_index(drop=True),y='Tích lũy',title='Đường cong tập trung doanh thu').show()
px.bar(bands,x='Dải ngày bán',y='Sản_phẩm',text='Sản_phẩm',title='Sản phẩm theo dải ngày bán').show()
px.bar(category.sort_values('recorded_sales_rate'),x='recorded_sales_rate',y='product_group',orientation='h',title='Hiệu quả theo nhóm sản phẩm').show()
px.bar(price,x='price_segment',y='recorded_sales_rate',title='Hiệu quả theo phân khúc giá').show()
px.bar(quality,x='Số sản phẩm',y='Chỉ tiêu',orientation='h',title='Chất lượng catalogue').show()
px.bar(tests,x='Yếu tố',y="Cramér's V",text_auto='.3f',title="Cường độ liên hệ Cramér's V").show()
folds=cv.iloc[:-1]; go.Figure([go.Scatter(x=folds.Fold,y=folds.AUC,name='AUC'),go.Scatter(x=folds.Fold,y=folds['Average precision'],name='Average precision')]).update_layout(title='Hồi quy logistic 5-fold').show()

In [ ]:
# Bước 5 - Ma trận nhóm sản phẩm × giá, MIWI và đánh giá
# 5.1. Tính tỷ lệ sản phẩm có bán theo nhóm và phân khúc giá
cell=(score.groupby(['product_group','price_segment'],as_index=False,dropna=False)
      .agg(active=('product_name_current','size'),sold=('recorded_sales','sum')))
cell['rate']=pd.to_numeric(cell['sold'],errors='coerce')/pd.to_numeric(cell['active'],errors='coerce')
matrix=cell.pivot(index='product_group',columns='price_segment',values='rate')
# BigQuery có thể trả cột nullable với pd.NA; Plotly chỉ nhận số thực/np.nan.
heatmap_values=matrix.to_numpy(dtype=float,na_value=np.nan)
go.Figure(go.Heatmap(z=heatmap_values,x=matrix.columns.astype(str).tolist(),y=matrix.index.astype(str).tolist(),colorscale='Greens',zmin=0,zmax=1,colorbar=dict(title='Tỷ lệ có bán'),hovertemplate='Nhóm: %{y}<br>Phân khúc giá: %{x}<br>Tỷ lệ có bán: %{z:.1%}<extra></extra>')).update_layout(title='Ma trận nhóm sản phẩm × phân khúc giá',xaxis_title='Phân khúc giá',yaxis_title='Nhóm sản phẩm',height=550).show()
# 5.2. Chuẩn hóa số liệu MIWI trước khi vẽ.
miwi_sum=(miwi[['missing_reported','wrong_reported']].apply(pd.to_numeric,errors='coerce').fillna(0).sum().rename('Số sự cố').reset_index().rename(columns={'index':'Loại sự cố'}))
px.bar(miwi_sum,x='Loại sự cố',y='Số sự cố',text='Số sự cố',title='Cơ cấu sự cố MIWI').show()
# 5.3. Chỉ vẽ các đánh giá có số sao hợp lệ.
reviews_chart=reviews.copy(); reviews_chart['rating']=pd.to_numeric(reviews_chart['rating'],errors='coerce')
px.histogram(reviews_chart.dropna(subset=['rating']),x='rating',nbins=5,title='Phân bố đánh giá khách hàng',labels={'rating':'Số sao'}).show()
# 5.4. Chuyển các tỷ trọng về kiểu số để kết quả nhất quán giữa CSV và BigQuery.
sens_chart=sens.copy()
for column in ['Tỷ trọng sản phẩm đã bán','Tỷ trọng doanh thu']:
    sens_chart[column]=pd.to_numeric(sens_chart[column],errors='coerce')
px.line(sens_chart,x='Ngưỡng ngày bán',y=['Tỷ trọng sản phẩm đã bán','Tỷ trọng doanh thu'],markers=True,title='Phân tích độ nhạy').show()
print('Bước 5 đã hoàn thành.')

In [ ]:
# Bước 6 - Xuất bảng kết quả để đối chiếu và tải về
OUT=Path('notebook_results'); OUT.mkdir(exist_ok=True)
outputs={'00_overview':overview,'01_business':business,'02_concentration':conc,'03_frequency':bands,'04_category':category,'05_price':price,'06_quality':quality,'07_promotions':promo,'08_chi_square_cramerv':tests,'09_logistic_5fold':cv,'10_sensitivity':sens,'11_operations':ops}
for name,frame in outputs.items(): frame.to_csv(OUT/f'{name}.csv',index=False,encoding='utf-8-sig')
print('Đã xuất',len(outputs),'bảng tại',OUT.resolve())
display(pd.DataFrame([['Tỷ lệ có bán',score.recorded_sales.astype(bool).mean()],['Gini toàn danh mục',conc.loc[conc['Chỉ tiêu'].str.contains('toàn danh mục'),'Giá trị'].iloc[0]],["Cramér's V nhóm",tests.loc[tests['Yếu tố'].eq('product_group'),"Cramér's V"].iloc[0]],['AUC trung bình',cv.loc[cv.Fold.eq('Trung bình'),'AUC'].iloc[0]]],columns=['Kết quả đối chiếu','Giá trị']))

## Cách kiểm chứng
Chạy tuần tự từ Bước 1 đến Bước 6. Các bảng CSV xuất trong `notebook_results` là kết quả trung gian để đối chiếu với báo cáo Word. Chi-square, Cramér's V và hồi quy logistic 5-fold được tính trực tiếp khi chạy notebook, không nhập tay.